# 26 · Dựng OCR cache ViTextVQA đã sửa EXIF (W14, bước 2 — phần A, T4)

**Vấn đề (notebook 25, luật R-EXIF):** **L1 XÁC NHẬN** — 164/165 ảnh có cờ xoay EXIF bị lệch cache, 0/200 ảnh đối chứng lệch; kiểu `asis` trùng cache 100% (lỗi tất định, không phải do GPU). Ở 126 câu test dùng các ảnh này, đáp án nằm trong OCR **0%** (cache) → **41,3%** (ảnh đã xoay). **L2:** 1,256% câu full test > 1% → phải sửa. **L3:** 1,28% câu train ≤ 5% → không train lại.

**Notebook này làm (luật ghi trước ở `docs/W14/README.md`, mục "R-EXIF bước 2"):**
1. Cài Paddle + VietOCR cùng phiên bản cache, kiểm engine (đã có bản sửa `oriented_image` trong `src/data/ocr.py`).
2. Chép cache W10 sang **file mới** `paddledet_vietocr_exiffix.json` và OCR lại **đúng** các ảnh có cờ xoay. Cache W10 **không bị đụng**.
3. Kiểm hợp lệ V1–V4 rồi ghi biên bản `experiments/W14_exif_fix/ocr_cache_exiffix_manifest.json`.

**Runtime:** **T4 GPU** (cùng loại GPU với lúc dựng cache W10, như notebook 25). Khoảng 20–25 phút. Cuối notebook tự trả máy.

**Anh chỉ cần:** Run all → gửi Claude **output mục 6**. Xong notebook này mới mở notebook 27 (L4).

## 1. Thiết lập môi trường

*Kỳ vọng: commit mới nhất (có `oriented_image` trong `src/data/ocr.py`), `Runtime: Tesla T4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, subprocess
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
assert 'def oriented_image' in open('src/data/ocr.py', encoding='utf-8').read(), '❌ Code cũ -> git pull chưa lấy được bản sửa EXIF'
assert 'R-EXIF bước 2' in open('docs/W14/README.md', encoding='utf-8').read(), '❌ Luật bước 2 chưa được commit -> DỪNG'

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
AUDIT = f'{DRIVE_ROOT}/experiments/W14_exif_audit_vitextvqa'
OUT = f'{DRIVE_ROOT}/experiments/W14_exif_fix'
os.makedirs(OUT, exist_ok=True)
OLD_CACHE = f'{DRIVE_ROOT}/data/vitextvqa_official/ocr/paddledet_vietocr.json'
NEW_CACHE = f'{DRIVE_ROOT}/data/vitextvqa_official/ocr/paddledet_vietocr_exiffix.json'
CACHE_SHA_W10 = 'e7ce4f76d98c1eb5dbbd7c07bb6ed62cfb3e4043b955143a2b5a6cc182560c1b'
for f in ('exif_scan.json', 'exif_reocr.json', 'exif_audit_summary.json'):
    assert os.path.exists(f'{AUDIT}/{f}'), f'❌ Thiếu {f} của notebook 25 trên Drive -> DỪNG'
V = json.load(open(f'{AUDIT}/exif_audit_summary.json', encoding='utf-8'))['verdict']
assert V['L1_mechanism_confirmed'] and V['L2'].startswith('> 1%'), V
print('Phán quyết notebook 25:', V['L1'], '|', V['L2'][:40], '…')

GPU = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip() or 'CPU'
print('Runtime:', GPU)
if 'T4' not in GPU:
    raise SystemExit('❌ Notebook 26 chạy trên T4 (cùng GPU với cache W10). Runtime -> Change runtime type -> T4 GPU, rồi Run all.')
print('✅ Thiết lập xong')

## 2. Kiểm thử offline + ảnh ViTextVQA

*Kỳ vọng: `exif_audit step2 OK`, `exif_audit SELFTEST OK`, `ocr exif fix OK`, `ocr SELFTEST OK`, dòng thư mục ảnh.*

In [ ]:
!python -W ignore -m src.data.exif_audit --selftest
!python -m src.data.ocr --selftest 2>&1 | tail -2
from src.eval.run_baseline import prepare_images
IMG_TV = prepare_images('vitextvqa_official', DRIVE_ROOT)
print('ảnh ViTextVQA:', IMG_TV)

## 3. Cài PaddleOCR 3.7 + VietOCR (như notebook 25) — Paddle **phải** chạy GPU

Cache W10 dựng bằng Paddle GPU, nên notebook này **dừng** nếu Paddle GPU không chạy được (không thay bằng CPU).

*Kỳ vọng: `paddleocr 3.7.0 | paddle 3.3.1`, `VietOCR import OK`, `DEVICE = gpu`. 3–6 phút.*

In [ ]:
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-1500:]

def nvidia_pkgs():
    out = subprocess.run('pip list --format=json', shell=True, capture_output=True, text=True).stdout
    return {p['name'].lower(): p['version'] for p in json.loads(out) if p['name'].lower().startswith('nvidia-')}

def torch_ok():
    return sh('python -c "import torch; assert torch.cuda.is_available(); torch.zeros(1).cuda()"')[0] == 0

def paddle_gpu_ok():
    return sh('python -c "import paddle; assert paddle.device.is_compiled_with_cuda(); paddle.utils.run_check()"')[0] == 0

def restore(pkgs):                       # cài lại ĐÚNG phiên bản cũ, không kéo thêm gì
    if pkgs:
        code_, log = sh('pip install -q --force-reinstall --no-deps ' + ' '.join(f'{n}=={v}' for n, v in pkgs.items()))
        assert code_ == 0, log

assert torch_ok(), '❌ torch đã hỏng từ trước -> Runtime -> Disconnect and delete runtime, rồi chạy lại từ mục 1'
print('torch OK trước khi cài')
code_, log = sh('pip install -q paddleocr==3.7.0')
assert code_ == 0, log
BEFORE = nvidia_pkgs()
for attempt in range(1, 4):
    code_gpu, log_gpu = sh('pip install -q --timeout 180 --retries 5 paddlepaddle-gpu==3.3.1 '
                           '-i https://www.paddlepaddle.org.cn/packages/stable/cu126/')   # = phiên bản cache ViTextVQA
    if code_gpu == 0:
        break
    print(f'⚠ Lần {attempt}/3: cài Paddle GPU lỗi -> thử lại. Đuôi log:', log_gpu[-200:].replace(chr(10), ' '))
NEW = sorted(set(nvidia_pkgs()) - set(BEFORE))
restore({n: v for n, v in BEFORE.items() if 'nccl' in n})
if not torch_ok():
    restore(BEFORE)
if code_gpu == 0 and torch_ok() and paddle_gpu_ok():
    DEVICE = 'gpu'
else:
    raise SystemExit('❌ Paddle GPU không chạy được trên T4 -> DỪNG (cache W10 dựng bằng GPU; không thay bằng CPU). '
                     'Gửi Claude output cell này.')
print(sh('python -c "import paddleocr, paddle; print(\'paddleocr\', paddleocr.__version__, \'| paddle\', paddle.__version__)"')[1])
code_, log = sh('pip install -q --no-deps vietocr==0.3.13 && pip install -q gdown einops')
assert code_ == 0, log
print(sh('python -c "from vietocr.tool.predictor import Predictor; print(\'VietOCR import OK\')"')[1])
print('DEVICE =', DEVICE, '| ✅ torch vẫn tốt')

## 4. Dựng cache đã sửa + kiểm hợp lệ

| Kiểm | Điều kiện |
|---|---|
| V1 | 0 ảnh OCR lỗi |
| V2 | mọi ảnh **không** cờ xoay giữ nguyên từng dòng của cache W10 |
| V3 | ≥ 95% ảnh mẫu bước B cho văn bản (160 ký tự đầu) **trùng** kiểu `fixed` của notebook 25 |
| V4 | engine trùng cache W10 ở `paddleocr`, `det_model`, `rec_model`, `vietocr_config`, `lang` **và** có khoá `exif` (bản sửa) |

*Kỳ vọng: `[R-EXIF fix] … 18x/18x`, rồi một dòng JSON có `"valid": true`, `n_replaced` ≈ 165–185, `parity` ≈ 100%, `pct_answer_in_ocr_test_fixed_images` old 0.0 → new ≈ 41. Khoảng 5 phút.*

In [ ]:
from src.data.exif_audit import sha256_file
assert sha256_file(OLD_CACHE) == CACHE_SHA_W10, '❌ Cache W10 trên Drive đã bị đổi -> DỪNG, báo Claude'
print('cache W10 sha256 trùng biên bản ✅')
if os.path.exists(f'{OUT}/ocr_cache_exiffix_manifest.json'):
    os.remove(f'{OUT}/ocr_cache_exiffix_manifest.json')   # không đọc nhầm biên bản cũ
!python -u -W ignore -m src.data.exif_audit --fix_cache --drive_root {DRIVE_ROOT} --scan_json {AUDIT}/exif_scan.json --reocr_json {AUDIT}/exif_reocr.json --vitextvqa_images {IMG_TV} --cache {OLD_CACHE} --new_cache {NEW_CACHE} --expect_cache_sha {CACHE_SHA_W10} --device {DEVICE} --out {OUT} 2>&1 | grep --line-buffered -v "ppocr\|Creating model\|Model files\|UserWarning"
MAN = json.load(open(f'{OUT}/ocr_cache_exiffix_manifest.json', encoding='utf-8'))
assert MAN['valid'], '❌ Cache đã sửa KHÔNG HỢP LỆ -> gửi Claude output mục 6'
assert sha256_file(OLD_CACHE) == CACHE_SHA_W10, '❌ Cache W10 bị ghi đè -> DỪNG'
print('✅ cache đã sửa hợp lệ, cache W10 vẫn nguyên')

## 5. Xem tận mắt 3 ảnh đã sửa

*Kỳ vọng: với mỗi ảnh, dòng "cũ" là chữ rác kiểu `0310000…`, dòng "mới" là chữ tiếng Việt đọc được.*

In [ ]:
from src.data.ocr import load_ocr_cache
from src.data.exif_audit import norm_text
old_c, new_c = load_ocr_cache(OLD_CACHE), load_ocr_cache(NEW_CACHE)
extra = [x for x in new_c['exif_fix']['replaced_images'] if x not in ('1849.jpg', '14277.jpg')][:1]
for k in ['1849.jpg', '14277.jpg'] + extra:
    print(f'— {k}\n  cũ: {norm_text(old_c["images"][k])[:110]}\n  mới: {norm_text(new_c["images"][k])[:110]}')

## 6. ⭐ Khối gửi Claude

*Kỳ vọng: khoảng 10 dòng, bắt đầu bằng `=== R-EXIF bước 2A ===`.*

In [ ]:
print('=== R-EXIF bước 2A ===')
!git log -1 --format='commit %h'
print('runtime', GPU, '| OCR device', DEVICE)
print(json.dumps({k: MAN[k] for k in ('valid', 'n_images', 'n_replaced', 'n_errors', 'V1_no_errors', 'V2_others_unchanged',
                                      'n_others', 'V3_parity_step_b', 'V4_engine', 'test_questions_on_fixed_images',
                                      'pct_answer_in_ocr_test_fixed_images', 'sec_per_image')}, ensure_ascii=False))
print('parity', json.dumps(MAN['parity'], ensure_ascii=False)[:600])
print('sha256 cache mới', MAN['sha256'])
print('engine', json.dumps({k: MAN['engine_info'].get(k) for k in ('paddleocr', 'paddlepaddle', 'det_model', 'rec_model', 'exif', 'device')}, ensure_ascii=False))

## 7. Trả máy

Kết quả: cache mới trên Drive `data/vitextvqa_official/ocr/paddledet_vietocr_exiffix.json`, biên bản ở `experiments/W14_exif_fix/`.

In [ ]:
AUTO_RELEASE = True
if AUTO_RELEASE:
    print('✅ Xong toàn bộ -> tự trả máy để ngừng tính CU (output phía trên vẫn giữ trong tab).')
    from google.colab import runtime
    runtime.unassign()
else:
    print('AUTO_RELEASE = False -> nhớ tự Disconnect and delete runtime khi xong.')